In [25]:
import nest_asyncio

nest_asyncio.apply()

import sys

sys.path.append("../")

from dotenv import load_dotenv

assert load_dotenv("../.env", override=True), "failed to load .env"

In [26]:
!docker exec crystalvision-ollama-1 ollama pull $OLLAMA_CODE_MODEL

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling d7e4b00a7d7a... 100% ▕████████████████▏  15 GB                         
pulling 109037bec39c... 100% ▕████████████████▏  136 B                         
pulling 097a36493f71... 100% ▕████████████████▏ 8.4 KB                         
pulling 2490e7468436... 100% ▕████████████████▏   65 B                         
pulling d109ef1844fa... 100% ▕████████████████▏  488 B                         
verifying sha256 digest 
writing manifest 
success 


In [27]:
import os
from crystalvision.lang.chat_model import MyChatOllama

llm = MyChatOllama(model=os.getenv("OLLAMA_CODE_MODEL"), temperature=0)

In [28]:
from crystalvision.lang.loaders import explain_database

df = explain_database()
df.head(3)

,code,rarity,cost,category_1,category_2,ex_burst,set,name_en,type_en,job_en,text_en,images,limit_break,icons,element_ja,power,element,numElements
0,1-001H,H,6,X,None,False,[Opus I],Auron,Forward,Guardian,"When Auron deals damage to your opponent, you may play 1 Fire Backup from your hand onto the field dull.",https://fftcg.cdn.sewest.net/images/cards/full/1-001H_eg.jpg,False,NaN,"(火,)",9000.0,{Fire},1
1,1-002R,R,5,X,None,False,[Opus I],Auron,Forward,Guardian,The Backups you control cannot be broken by your opponent's Summons or abilities.,https://fftcg.cdn.sewest.net/images/cards/full/1-002R_eg.jpg,False,NaN,"(火,)",9000.0,{Fire},1
2,1-003C,C,2,III,None,False,[Opus I],Red Mage,Backup,Standard Unit,《火》《1》《ダル》: Choose 1 Forward. It cannot block this turn.,https://fftcg.cdn.sewest.net/images/cards/full/1-003C_eg.jpg,False,multicard,"(火,)",NaN,{Fire},1


In [29]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [30]:
from langchain_experimental.tools.python.tool import PythonAstREPLTool
from crystalvision.lang.tools import MultiImageEmbedTool

tools = [PythonAstREPLTool(locals={"df": df}), MultiImageEmbedTool(df)]

In [31]:
from langchain_core.globals import set_debug

set_debug(False)

In [32]:
from langchain_core.messages import ChatMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


prompt = ChatPromptTemplate(
    [
        (
            "system",
            """You are working with a pandas dataframe in Python. The name of the dataframe is `df`.

In the dataframe, some of the columns are as follows:

{column_description}

You should use the tools below to answer the question posed of you:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

The previous conversation history:
""",
        ),
        # SystemMessagePromptTemplate(prompt=sysmsg),
        # SystemMessage(sysmsg.template, input_variables=sysmsg.input_variables),
        MessagesPlaceholder(variable_name="history", optional=True),
        # ("user", "\nBegin!\nQuestion: {input}\n{agent_scratchpad}")
        (
            "human",
            "\nBegin!\nI am {username} and my Question is: {input}\n{agent_scratchpad}",
        ),
        # HumanMessage("\nBegin!\n{role}'s Question: {input}\n{agent_scratchpad}", name="Tacoking"), #not quite working
        # ("{role}", "\nBegin!\nQuestion: {input}\n{agent_scratchpad}")
        # ChatMessage("\nBegin!\nQuestion: {input}\n{agent_scratchpad}", role="{role}")
    ]
)
prompt.pretty_print()

================================ System Message ================================

You are working with a pandas dataframe in Python. The name of the dataframe is `df`.

In the dataframe, some of the columns are as follows:

{column_description}

You should use the tools below to answer the question posed of you:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

The previous conversation history:


============================= Messages Placeholder =============================

{history}

================================ Human Message =================================


Begin!
I am {username} and my Quest

In [33]:
from langchain_core.tools.render import render_text_description

column_description = ""
for col in df.columns:
    if col_desc := df[col].attrs.get("description", ""):
        column_description += f"'{col}' refers to {col_desc}.\n"

pprompt = prompt.partial(
    tools=render_text_description(tools),
    tool_names=", ".join([t.name for t in tools]),
    column_description=column_description,
    # agent_scratchpad="",
)
print(
    pprompt.format(
        input="This is a test input",
        agent_scratchpad="[agent scratchpad placeholder]",
        username="tacoking",
    )
)

System: You are working with a pandas dataframe in Python. The name of the dataframe is `df`.

In the dataframe, some of the columns are as follows:

'code' refers to the UUID of the card.
'ex_burst' refers to if the card has an exburst/ex burst/ex-burst/ex/EX ability.
'name_en' refers to the English name of the card.
'type_en' refers to the English type of the card.
'images' refers to the card image URL.
'element_ja' refers to each element of the card in Japanese as python set.
'power' refers to the numerical value of power of the card.
'element' refers to each element of the card as python set.
'numElements' refers to the number of elements the card consists of.


You should use the tools below to answer the question posed of you:

python_repl_ast - A Python shell. Use this to execute python commands. Input should be a valid python command. When using this tool, sometimes output is abbreviated - make sure it does not look abbreviated before using it in your answer.
MultiImageEmbedToo

In [34]:
from langchain.agents.agent import AgentExecutor, BaseSingleActionAgent, RunnableAgent
from langchain.agents import (
    create_react_agent,
)

agent: BaseSingleActionAgent = RunnableAgent(
    runnable=create_react_agent(llm, tools, pprompt),  # type: ignore
    input_keys_arg=["input"],
    return_keys_arg=["output"],
    stream_runnable=False,
)

agent = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    # **(agent_executor_kwargs or {}),
)

In [35]:
from crystalvision.lang.chat_cache import get_message_history

history = get_message_history("test-0")
history.clear()

In [36]:
history2 = get_message_history("test-1")
history2.clear()

In [37]:
agent.invoke({"input": "Tell me a random card.", "username": "test"})



> Entering new AgentExecutor chain...
Thought: I need to randomly select a row from the dataframe.
Action: python_repl_ast
Action Input: `df.sample()`        code rarity  cost category_1 category_2  ex_burst        set name_en  \
2553  4-028C      C     2        FFT       None     False  [Opus IV]    Bard   

     type_en         job_en  \
2553  Backup  Standard Unit   

                                                                                                                                             text_en  \
2553  《氷》《氷》《2》《ダル》, put Bard into the Break Zone: Your opponent discards 2 cards from his/her hand. You can only use this ability during your turn.   

                                                            images  \
2553  https://fftcg.cdn.sewest.net/images/cards/full/4-028C_eg.jpg   

      limit_break      icons element_ja  power element  numElements  
2553        False  multicard       (氷,)    NaN   {Ice}            1  I now know the final answer
Final Answe

{'input': 'Tell me a random card.',
 'username': 'test',
 'output': "The card Bard (4-028C) is a Backup Standard Unit with the ability to discard 2 cards from your opponent's hand. It costs 2 CP and has the element Ice."}

In [38]:
from crystalvision.lang.chat_cache import MyRunnableWithMessageHistory


hagent: BaseSingleActionAgent = MyRunnableWithMessageHistory(
    agent,
    get_message_history,
    input_messages_key="input",
    output_messages_key="output",
    history_messages_key="history",
    # history_factory_config=[
    #     ConfigurableFieldSpec(
    #         id="user_id",
    #         annotation=str,
    #         name="User ID",
    #         description="Unique identifier for the user.",
    #         default="",
    #         is_shared=True,
    #     ),
    # ]
)

In [39]:
hagent.invoke(
    {
        "input": "My favorite card is 1-001H, please tell me more about it.",
        "username": "tacoking",
    },
    config={"configurable": {"session_id": "test-0"}},
)



> Entering new AgentExecutor chain...
Question: My favorite card is 1-001H, please tell me more about it.
Thought: I need to find the row in the dataframe where 'code' is '1-001H'. Then I can print information about that card.
Action: python_repl_ast
Action Input: `df[df['code'] == '1-001H']`     code rarity  cost category_1 category_2  ex_burst       set name_en  \
0  1-001H      H     6          X       None     False  [Opus I]   Auron   

   type_en    job_en  \
0  Forward  Guardian   

                                                                                                    text_en  \
0  When Auron deals damage to your opponent, you may play 1 Fire Backup from your hand onto the field dull.   

                                                         images  limit_break  \
0  https://fftcg.cdn.sewest.net/images/cards/full/1-001H_eg.jpg        False   

  icons element_ja   power element  numElements  
0   NaN       (火,)  9000.0  {Fire}            1  I now know the final

{'input': 'My favorite card is 1-001H, please tell me more about it.',
 'username': 'tacoking',
 'history': [],
 'output': 'The card 1-001H is named Auron, a Forward Guardian type card from Opus I. It costs 6 CP and has a power of 9000. Its ability lets you play a Fire Backup from your hand onto the field dull when it deals damage to your opponent.  It only has one element: Fire.'}

In [40]:
hagent.invoke(
    {
        "input": "Tell me random card named Zidane. This will be my favorite card.",
        "username": "tonberryking",
    },
    config={"configurable": {"session_id": "test-0"}},
)



> Entering new AgentExecutor chain...
Question: Tell me random card named Zidane. This will be my favorite card.
Thought: I need to find a Zidane card in the dataframe.
Action: python_repl_ast
Action Input: ```python
df[df['name_en'].str.contains('Zidane')].sample() 
```        code rarity  cost category_1 category_2  ex_burst        set name_en  \
363  11-008H      H     2         IX       None     False  [Opus XI]  Zidane   

     type_en job_en  \
363  Forward  Thief   

                                                                                                          text_en  \
363  Haste    When Zidane deals damage to your opponent, choose 1 Forward opponent controls. Deal it 8000 damage.   

                                                            images  \
363  https://fftcg.cdn.sewest.net/images/cards/full/11-008H_eg.jpg   

     limit_break icons element_ja   power element  numElements  
363        False   NaN       (火,)  2000.0  {Fire}            1  I found a Zida

{'input': 'Tell me random card named Zidane. This will be my favorite card.',
 'username': 'tonberryking',
 'history': [ChatMessage(content='My favorite card is 1-001H, please tell me more about it.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:05.001348'),
  AIMessage(content='The card 1-001H is named Auron, a Forward Guardian type card from Opus I. It costs 6 CP and has a power of 9000. Its ability lets you play a Fire Backup from your hand onto the field dull when it deals damage to your opponent.  It only has one element: Fire.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:05.001632')],
 'output': "Your new favorite card is Zidane (11-008H)!\n\nIt's a Forward, Thief type card from Opus XI with 2 CP and a power of 2000. It has one element: Fire.\n\nIts ability lets you deal 8000 damage to a Forward your opponent controls when Zidane deals damage to them!"}

In [41]:
hagent.invoke(
    {"input": "Tell me about the attributes of 1-177R.", "username": "tacoking"},
    config={"configurable": {"session_id": "test-1"}},
)



> Entering new AgentExecutor chain...
Question: Tell me about the attributes of 1-177R.
Thought: I need to find the row in the dataframe where 'code' is '1-177R' and then display its attributes.
Action: python_repl_ast
Action Input: `df[df['code'] == '1-177R'].to_dict('records')`[{'code': '1-177R', 'rarity': 'R', 'cost': 2, 'category_1': 'X', 'category_2': None, 'ex_burst': False, 'set': ['Opus I'], 'name_en': 'Yuna', 'type_en': 'Backup', 'job_en': 'Summoner', 'text_en': 'The cost required to cast your Water Summons is reduced by 1 (it cannot become 0).', 'images': 'https://fftcg.cdn.sewest.net/images/cards/full/1-177R_eg.jpg', 'limit_break': False, 'icons': nan, 'element_ja': ('水',), 'power': nan, 'element': {'Water'}, 'numElements': 1}]I now know the final answer
Final Answer: The card 1-177R is named Yuna and is a Backup type card. It belongs to the Summoner job and reduces the cost of casting Water Summons by 1. Its element is Water.  It has a rarity of R and costs 2 CP.

> Finis

{'input': 'Tell me about the attributes of 1-177R.',
 'username': 'tacoking',
 'history': [],
 'output': 'The card 1-177R is named Yuna and is a Backup type card. It belongs to the Summoner job and reduces the cost of casting Water Summons by 1. Its element is Water.  It has a rarity of R and costs 2 CP.'}

In [42]:
hagent.invoke(
    {
        "input": "Tell me a single card with a different code but the same name.",
        "username": "tacoking",
    },
    config={"configurable": {"session_id": "test-0"}},
)



> Entering new AgentExecutor chain...
Question: Tell me a single card with a different code but the same name.
Thought: I need to find a card with the same name as Zidane but a different code.
Action: python_repl_ast
Action Input: 
```python
df[df['name_en'] == 'Zidane'].code.to_list()
```['1-071L', '10-051C', '11-008H', '14-127H', '16-048H', '19-108L', '23-008H', '24-044H', '24-117R', '3-056H', '3-154S', '6-044L', '8-115L']I now know the final answer.

Final Answer:  The card Zidane (10-051C) is a different version of Zidane with a different code. 




> Finished chain.


{'input': 'Tell me a single card with a different code but the same name.',
 'username': 'tacoking',
 'history': [ChatMessage(content='My favorite card is 1-001H, please tell me more about it.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:05.001348'),
  AIMessage(content='The card 1-001H is named Auron, a Forward Guardian type card from Opus I. It costs 6 CP and has a power of 9000. Its ability lets you play a Fire Backup from your hand onto the field dull when it deals damage to your opponent.  It only has one element: Fire.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:05.001632'),
  ChatMessage(content='Tell me random card named Zidane. This will be my favorite card.', additional_kwargs={}, response_metadata={}, role='User (tonberryking)', timestamp='2024-12-13T12:25:10.918668'),
  AIMessage(content="Your new favorite card is Zidane (11-008H)!\n\nIt's a Forward, Thief type card from Opus XI with 2 CP and

In [43]:
hagent.invoke(
    {
        "input": "Tell me about another different card with the same name. A different card means not the same code.",
        "username": "tacoking",
    },
    config={"configurable": {"session_id": "test-1"}},
)



> Entering new AgentExecutor chain...
Thought: I need to find another card with the name "Yuna".  Since 'name_en' refers to the English name, I should filter the dataframe for rows where 'name_en' is "Yuna" and then check if the 'code' is different from 1-177R.
Action: python_repl_ast
Action Input: ```python
df[df['name_en'] == 'Yuna'][['code','name_en','type_en']]
```         code name_en  type_en
175    1-176H    Yuna   Backup
176    1-177R    Yuna   Backup
213    1-214S    Yuna  Forward
416   11-061L    Yuna  Forward
600   12-105L    Yuna  Forward
1165  16-134S    Yuna  Forward
1334  18-033R    Yuna  Forward
1539  19-098C    Yuna   Backup
1559  19-118L    Yuna  Forward
1717   2-138L    Yuna  Forward
1844  20-117L    Yuna  Forward
2097  22-106R    Yuna  Forward
2962   6-124L    Yuna  Forward
3095   7-127L    Yuna  ForwardI see there are multiple cards named "Yuna". I need to pick one that has a different 'code' than 1-177R.

Action: python_repl_ast
Action Input: ```python
df[(df['n

{'input': 'Tell me about another different card with the same name. A different card means not the same code.',
 'username': 'tacoking',
 'history': [ChatMessage(content='Tell me about the attributes of 1-177R.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:15.339999'),
  AIMessage(content='The card 1-177R is named Yuna and is a Backup type card. It belongs to the Summoner job and reduces the cost of casting Water Summons by 1. Its element is Water.  It has a rarity of R and costs 2 CP.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:15.340087')],
 'output': 'Another card named "Yuna" is 1-176H. It is a Backup type card.'}

In [44]:
hagent.invoke(
    {
        "input": "Tell me about another single card with the same name, a different code, and a different element.",
        "username": "tacoking",
    },
    config={"configurable": {"session_id": "test-0"}},
)



> Entering new AgentExecutor chain...
Question: Tell me about another single card with the same name, a different code, and a different element.
Thought: I need to find a Zidane card with a different code and element than 10-051C (Fire).
Action: python_repl_ast
Action Input: ```python
df[df['name_en'] == 'Zidane'][['code', 'element']]
```         code        element
70     1-071L         {Wind}
266   10-051C         {Wind}
363   11-008H         {Fire}
888   14-127H  {Wind, Water}
1079  16-048H         {Wind}
1549  19-108L   {Fire, Wind}
2123  23-008H         {Fire}
2289  24-044H         {Wind}
2362  24-117R         {Wind}
2427   3-056H         {Wind}
2525   3-154S        {Light}
2882   6-044L         {Wind}
3221   8-115L        {Water}I found it! Zidane (3-154S) is a Light element card. 

Thought: I now know the final answer
Final Answer: Zidane (3-154S) is a Light element card.

> Finished chain.


{'input': 'Tell me about another single card with the same name, a different code, and a different element.',
 'username': 'tacoking',
 'history': [ChatMessage(content='My favorite card is 1-001H, please tell me more about it.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:05.001348'),
  AIMessage(content='The card 1-001H is named Auron, a Forward Guardian type card from Opus I. It costs 6 CP and has a power of 9000. Its ability lets you play a Fire Backup from your hand onto the field dull when it deals damage to your opponent.  It only has one element: Fire.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:05.001632'),
  ChatMessage(content='Tell me random card named Zidane. This will be my favorite card.', additional_kwargs={}, response_metadata={}, role='User (tonberryking)', timestamp='2024-12-13T12:25:10.918668'),
  AIMessage(content="Your new favorite card is Zidane (11-008H)!\n\nIt's a Forward, Thief ty

In [45]:
hagent.invoke(
    {
        "input": "Tell me about another different card with the same name and a different element. A different card means not the same code.",
        "username": "tacoking",
    },
    config={"configurable": {"session_id": "test-1"}},
)



> Entering new AgentExecutor chain...
Thought: I need to find a card named "Yuna" that has a different element than the ones I already know.
Action: python_repl_ast
Action Input: 
```python
df[df['name_en'] == 'Yuna'][['element', 'code']]
```            element     code
175         {Water}   1-176H
176         {Water}   1-177R
213         {Water}   1-214S
416          {Wind}  11-061L
600         {Water}  12-105L
1165         {Wind}  16-134S
1334          {Ice}  18-033R
1539        {Water}  19-098C
1559  {Wind, Water}  19-118L
1717        {Water}   2-138L
1844        {Water}  20-117L
2097        {Water}  22-106R
2962        {Water}   6-124L
3095        {Light}   7-127LI found a "Yuna" card with the element Ice. 

Thought: I now know the final answer
Final Answer: There is a Yuna card with the code 18-033R and the element Ice.

> Finished chain.


{'input': 'Tell me about another different card with the same name and a different element. A different card means not the same code.',
 'username': 'tacoking',
 'history': [ChatMessage(content='Tell me about the attributes of 1-177R.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:15.339999'),
  AIMessage(content='The card 1-177R is named Yuna and is a Backup type card. It belongs to the Summoner job and reduces the cost of casting Water Summons by 1. Its element is Water.  It has a rarity of R and costs 2 CP.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:15.340087'),
  ChatMessage(content='Tell me about another different card with the same name. A different card means not the same code.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:26.791312'),
  AIMessage(content='Another card named "Yuna" is 1-176H. It is a Backup type card.', additional_kwargs={}, respo

In [46]:
hagent.invoke(
    {"input": "Again please.", "username": "tacoking"},
    config={"configurable": {"session_id": "test-1"}},
)



> Entering new AgentExecutor chain...
Thought: I need to find another Yuna card with a different element than Water or Ice.
Action: python_repl_ast
Action Input: 
```python
df[df['name_en'] == 'Yuna']['element'].unique()
```TypeError: unhashable type: 'set'I need to convert the sets in the 'element' column to a hashable type like strings.

Action: python_repl_ast
Action Input: 
```python
df['element_str'] = df['element'].apply(lambda x: ','.join(sorted(list(x))))
df[df['name_en'] == 'Yuna']['element_str'].unique()
```['Water' 'Wind' 'Ice' 'Water,Wind' 'Light']I now know the final answer.

Final Answer: There is a Yuna card with the element Wind. 




> Finished chain.


{'input': 'Again please.',
 'username': 'tacoking',
 'history': [ChatMessage(content='Tell me about the attributes of 1-177R.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:15.339999'),
  AIMessage(content='The card 1-177R is named Yuna and is a Backup type card. It belongs to the Summoner job and reduces the cost of casting Water Summons by 1. Its element is Water.  It has a rarity of R and costs 2 CP.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:15.340087'),
  ChatMessage(content='Tell me about another different card with the same name. A different card means not the same code.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:26.791312'),
  AIMessage(content='Another card named "Yuna" is 1-176H. It is a Backup type card.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:26.791426'),
  ChatMessage(content='Tell me about another differ

In [47]:
hagent.invoke(
    {"input": "What about the rest? Pick up to 5.", "username": "tacoking"},
    config={"configurable": {"session_id": "test-1"}},
)



> Entering new AgentExecutor chain...
Thought: I need to find other Yuna cards besides the ones already mentioned.
Action: python_repl_ast
Action Input: ```python
df[df['name_en'] == 'Yuna'][['code', 'element']]
```         code        element
175    1-176H        {Water}
176    1-177R        {Water}
213    1-214S        {Water}
416   11-061L         {Wind}
600   12-105L        {Water}
1165  16-134S         {Wind}
1334  18-033R          {Ice}
1539  19-098C        {Water}
1559  19-118L  {Wind, Water}
1717   2-138L        {Water}
1844  20-117L        {Water}
2097  22-106R        {Water}
2962   6-124L        {Water}
3095   7-127L        {Light}I now know the final answer
Final Answer: There are other Yuna cards with the elements Wind, Ice, and Light. 

There is also a Yuna card that has both Water and Wind as its element.

> Finished chain.


{'input': 'What about the rest? Pick up to 5.',
 'username': 'tacoking',
 'history': [ChatMessage(content='Tell me about the attributes of 1-177R.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:15.339999'),
  AIMessage(content='The card 1-177R is named Yuna and is a Backup type card. It belongs to the Summoner job and reduces the cost of casting Water Summons by 1. Its element is Water.  It has a rarity of R and costs 2 CP.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:15.340087'),
  ChatMessage(content='Tell me about another different card with the same name. A different card means not the same code.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:26.791312'),
  AIMessage(content='Another card named "Yuna" is 1-176H. It is a Backup type card.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:26.791426'),
  ChatMessage(content='Tell me

In [48]:
hagent.invoke(
    {
        "input": "Tell me about any other cards with the same name you have not mentioned yet please.",
        "username": "tacoking",
    },
    config={"configurable": {"session_id": "test-1"}},
)



> Entering new AgentExecutor chain...
Thought: I need to find more Yuna cards in the dataframe.
Action: python_repl_ast
Action Input: 
```python
df[df['name_en'] == 'Yuna']['code'].tolist()
```['1-176H', '1-177R', '1-214S', '11-061L', '12-105L', '16-134S', '18-033R', '19-098C', '19-118L', '2-138L', '20-117L', '22-106R', '6-124L', '7-127L']I have found more Yuna cards.

Thought: I need to find the element of each card and display it.
Action: python_repl_ast
Action Input: 
```python
for code in ['1-176H', '1-177R', '1-214S', '11-061L', '12-105L', '16-134S', '18-033R', '19-098C', '19-118L', '2-138L', '20-117L', '22-106R', '6-124L', '7-127L']:
  print(f"{code}: {df[df['code'] == code]['element'].iloc[0]}") 
```1-176H: {'Water'}
1-177R: {'Water'}
1-214S: {'Water'}
11-061L: {'Wind'}
12-105L: {'Water'}
16-134S: {'Wind'}
18-033R: {'Ice'}
19-098C: {'Water'}
19-118L: {'Wind', 'Water'}
2-138L: {'Water'}
20-117L: {'Water'}
22-106R: {'Water'}
6-124L: {'Water'}
7-127L: {'Light'}
I now know the fin

{'input': 'Tell me about any other cards with the same name you have not mentioned yet please.',
 'username': 'tacoking',
 'history': [ChatMessage(content='Tell me about the attributes of 1-177R.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:15.339999'),
  AIMessage(content='The card 1-177R is named Yuna and is a Backup type card. It belongs to the Summoner job and reduces the cost of casting Water Summons by 1. Its element is Water.  It has a rarity of R and costs 2 CP.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:15.340087'),
  ChatMessage(content='Tell me about another different card with the same name. A different card means not the same code.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:26.791312'),
  AIMessage(content='Another card named "Yuna" is 1-176H. It is a Backup type card.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T

In [49]:
hagent.invoke(
    {"input": "What is the text_en of my favorite card?", "username": "tacoking"},
    config={"configurable": {"session_id": "test-0"}},
)



> Entering new AgentExecutor chain...
Thought: I need to find the 'text_en' column for Zidane (11-008H).
Action: python_repl_ast
Action Input: 
```python
df[df['name_en'] == 'Zidane'][df['code'] == '11-008H']['text_en'].values[0]
```Haste    When Zidane deals damage to your opponent, choose 1 Forward opponent controls. Deal it 8000 damage.

<string>:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


I now know the final answer
Final Answer: The text_en of my favorite card is "Haste\n When Zidane deals damage to your opponent, choose 1 Forward opponent controls. Deal it 8000 damage."

> Finished chain.


{'input': 'What is the text_en of my favorite card?',
 'username': 'tacoking',
 'history': [ChatMessage(content='My favorite card is 1-001H, please tell me more about it.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:05.001348'),
  AIMessage(content='The card 1-001H is named Auron, a Forward Guardian type card from Opus I. It costs 6 CP and has a power of 9000. Its ability lets you play a Fire Backup from your hand onto the field dull when it deals damage to your opponent.  It only has one element: Fire.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:05.001632'),
  ChatMessage(content='Tell me random card named Zidane. This will be my favorite card.', additional_kwargs={}, response_metadata={}, role='User (tonberryking)', timestamp='2024-12-13T12:25:10.918668'),
  AIMessage(content="Your new favorite card is Zidane (11-008H)!\n\nIt's a Forward, Thief type card from Opus XI with 2 CP and a power of 2000. It h

In [50]:
hagent.invoke(
    {"input": "What is the text_en of my favorite card?", "username": "tonberryking"},
    config={"configurable": {"session_id": "test-0"}},
)



> Entering new AgentExecutor chain...
Question: What is the text_en of my favorite card?
Thought: I need to know which card is your favorite.
Action: python_repl_ast
Action Input: print(df[df['name_en'] == 'Zidane']['text_en'].iloc[0])Zidane cannot be chosen by your opponent's Summons or abilities.  Zidane cannot be blocked by a Forward of cost 4 or more.
I now know the final answer
Final Answer: Zidane cannot be chosen by your opponent's Summons or abilities.
Zidane cannot be blocked by a Forward of cost 4 or more.

> Finished chain.


{'input': 'What is the text_en of my favorite card?',
 'username': 'tonberryking',
 'history': [ChatMessage(content='My favorite card is 1-001H, please tell me more about it.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:05.001348'),
  AIMessage(content='The card 1-001H is named Auron, a Forward Guardian type card from Opus I. It costs 6 CP and has a power of 9000. Its ability lets you play a Fire Backup from your hand onto the field dull when it deals damage to your opponent.  It only has one element: Fire.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:05.001632'),
  ChatMessage(content='Tell me random card named Zidane. This will be my favorite card.', additional_kwargs={}, response_metadata={}, role='User (tonberryking)', timestamp='2024-12-13T12:25:10.918668'),
  AIMessage(content="Your new favorite card is Zidane (11-008H)!\n\nIt's a Forward, Thief type card from Opus XI with 2 CP and a power of 2000. 

In [51]:
# hagent.invoke({"input": "Who am I? What was my last question?", "username": "tonberryking"}, config={"configurable": {"session_id": "test-0"}})

In [52]:
history.messages

[ChatMessage(content='My favorite card is 1-001H, please tell me more about it.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:05.001348'),
 AIMessage(content='The card 1-001H is named Auron, a Forward Guardian type card from Opus I. It costs 6 CP and has a power of 9000. Its ability lets you play a Fire Backup from your hand onto the field dull when it deals damage to your opponent.  It only has one element: Fire.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:05.001632'),
 ChatMessage(content='Tell me random card named Zidane. This will be my favorite card.', additional_kwargs={}, response_metadata={}, role='User (tonberryking)', timestamp='2024-12-13T12:25:10.918668'),
 AIMessage(content="Your new favorite card is Zidane (11-008H)!\n\nIt's a Forward, Thief type card from Opus XI with 2 CP and a power of 2000. It has one element: Fire.\n\nIts ability lets you deal 8000 damage to a Forward your opponent cont

In [53]:
from langchain_core.prompt_values import ChatPromptValue

ChatPromptValue(messages=[ChatMessage(content="hmmm", role="tacoking")]).to_string()

'tacoking: hmmm'

In [54]:
print(ChatPromptValue(messages=history.messages).to_string())

User (tacoking): My favorite card is 1-001H, please tell me more about it.
AI: The card 1-001H is named Auron, a Forward Guardian type card from Opus I. It costs 6 CP and has a power of 9000. Its ability lets you play a Fire Backup from your hand onto the field dull when it deals damage to your opponent.  It only has one element: Fire.
User (tonberryking): Tell me random card named Zidane. This will be my favorite card.
AI: Your new favorite card is Zidane (11-008H)!

It's a Forward, Thief type card from Opus XI with 2 CP and a power of 2000. It has one element: Fire.

Its ability lets you deal 8000 damage to a Forward your opponent controls when Zidane deals damage to them!
User (tacoking): Tell me a single card with a different code but the same name.
AI: The card Zidane (10-051C) is a different version of Zidane with a different code.
User (tacoking): Tell me about another single card with the same name, a different code, and a different element.
AI: Zidane (3-154S) is a Light eleme

In [55]:
history2.messages

[ChatMessage(content='Tell me about the attributes of 1-177R.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:15.339999'),
 AIMessage(content='The card 1-177R is named Yuna and is a Backup type card. It belongs to the Summoner job and reduces the cost of casting Water Summons by 1. Its element is Water.  It has a rarity of R and costs 2 CP.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:15.340087'),
 ChatMessage(content='Tell me about another different card with the same name. A different card means not the same code.', additional_kwargs={}, response_metadata={}, role='User (tacoking)', timestamp='2024-12-13T12:25:26.791312'),
 AIMessage(content='Another card named "Yuna" is 1-176H. It is a Backup type card.', additional_kwargs={}, response_metadata={}, timestamp='2024-12-13T12:25:26.791426'),
 ChatMessage(content='Tell me about another different card with the same name and a different element. A different car

In [56]:
print(ChatPromptValue(messages=history2.messages).to_string())

User (tacoking): Tell me about the attributes of 1-177R.
AI: The card 1-177R is named Yuna and is a Backup type card. It belongs to the Summoner job and reduces the cost of casting Water Summons by 1. Its element is Water.  It has a rarity of R and costs 2 CP.
User (tacoking): Tell me about another different card with the same name. A different card means not the same code.
AI: Another card named "Yuna" is 1-176H. It is a Backup type card.
User (tacoking): Tell me about another different card with the same name and a different element. A different card means not the same code.
AI: There is a Yuna card with the code 18-033R and the element Ice.
User (tacoking): Again please.
AI: There is a Yuna card with the element Wind.
User (tacoking): What about the rest? Pick up to 5.
AI: There are other Yuna cards with the elements Wind, Ice, and Light. 

There is also a Yuna card that has both Water and Wind as its element.
User (tacoking): Tell me about any other cards with the same name you hav

In [57]:
print(
    pprompt.format(
        input="This is a test input",
        agent_scratchpad="[agent scratchpad placeholder]",
        username="tacoking",
        history=history.messages,
    )
)

System: You are working with a pandas dataframe in Python. The name of the dataframe is `df`.

In the dataframe, some of the columns are as follows:

'code' refers to the UUID of the card.
'ex_burst' refers to if the card has an exburst/ex burst/ex-burst/ex/EX ability.
'name_en' refers to the English name of the card.
'type_en' refers to the English type of the card.
'images' refers to the card image URL.
'element_ja' refers to each element of the card in Japanese as python set.
'power' refers to the numerical value of power of the card.
'element' refers to each element of the card as python set.
'numElements' refers to the number of elements the card consists of.


You should use the tools below to answer the question posed of you:

python_repl_ast - A Python shell. Use this to execute python commands. Input should be a valid python command. When using this tool, sometimes output is abbreviated - make sure it does not look abbreviated before using it in your answer.
MultiImageEmbedToo